In [ ]:

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import csv

from tensorflow.keras import layers, models

# ==========================================
# 1. LOAD MNIST DATASET
# ==========================================

data = np.load("mnist.npz")

x_train = data["x_train"]
y_train = data["y_train"]

x_test = data["x_test"]
y_test = data["y_test"]

# Normalize images
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = y_train.reshape(-1)
y_test = y_test.reshape(-1)

# Handle one-hot encoded labels
if y_train.ndim > 1:
    y_train = np.argmax(y_train, axis=1)

if y_test.ndim > 1:
    y_test = np.argmax(y_test, axis=1)

# Add channel dimension
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

print("Training data:", x_train.shape)
print("Testing data:", x_test.shape)


# ==========================================
# 2. BUILD CNN MODEL
# ==========================================

def create_model(learning_rate, num_layers):

    model = models.Sequential()

    model.add(layers.Input(shape=(28, 28, 1)))

    # Add convolution layers
    for i in range(num_layers):

        filters = 32 * (i + 1)

        model.add(
            layers.Conv2D(
                filters,
                (3, 3),
                activation="relu",
                padding="same"
            )
        )

        model.add(layers.MaxPooling2D((2, 2)))

    # Flatten and dense layers
    model.add(layers.Flatten())

    model.add(layers.Dense(128, activation="relu"))

    model.add(layers.Dense(10, activation="softmax"))

    # Compile model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ==========================================
# 3. DEFINE HYPERPARAMETERS
# ==========================================

configurations = [
    {
        "name": "Experiment_1",
        "learning_rate": 0.001,
        "batch_size": 32,
        "num_layers": 1
    },

    {
        "name": "Experiment_2",
        "learning_rate": 0.0001,
        "batch_size": 32,
        "num_layers": 2
    },

    {
        "name": "Experiment_3",
        "learning_rate": 0.001,
        "batch_size": 64,
        "num_layers": 2
    },

    {
        "name": "Experiment_4",
        "learning_rate": 0.01,
        "batch_size": 64,
        "num_layers": 2
    }
]


# ==========================================
# 4. TRAIN AND COMPARE MODELS
# ==========================================

results = []
histories = {}

epochs = 5

for config in configurations:

    print("\n================================")
    print("Running:", config["name"])
    print("================================")

    # Create model
    model = create_model(
        config["learning_rate"],
        config["num_layers"]
    )

    # TensorBoard log directory
    log_dir = os.path.join(
        "logs",
        config["name"]
    )

    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=0
    )

    # Measure training time
    start_time = time.time()

    # Train model
    history = model.fit(
        x_train,
        y_train,
        epochs=epochs,
        batch_size=config["batch_size"],
        validation_split=0.1,
        callbacks=[tensorboard_callback],
        verbose=1
    )

    training_time = time.time() - start_time

    # Evaluate model
    test_loss, test_accuracy = model.evaluate(
        x_test,
        y_test,
        verbose=0
    )

    # Store results
    histories[config["name"]] = history

    results.append({
        "Experiment": config["name"],
        "Learning Rate": config["learning_rate"],
        "Batch Size": config["batch_size"],
        "Layers": config["num_layers"],
        "Accuracy": round(test_accuracy * 100, 2),
        "Loss": round(test_loss, 4),
        "Training Time": round(training_time, 2)
    })

    print("\nTest Accuracy:", test_accuracy * 100, "%")
    print("Test Loss:", test_loss)
    print("Training Time:", training_time, "seconds")


# ==========================================
# 5. DISPLAY COMPARISON RESULTS
# ==========================================

print("\n========== HYPERPARAMETER COMPARISON ==========")

for result in results:
    print(result)


# ==========================================
# 6. SAVE RESULTS TO CSV
# ==========================================

with open("tuning_results.csv", "w", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=results[0].keys()
    )

    writer.writeheader()
    writer.writerows(results)

print("\nResults saved to tuning_results.csv")


# ==========================================
# 7. PLOT ACCURACY COMPARISON
# ==========================================

plt.figure(figsize=(10, 5))

for name, history in histories.items():

    plt.plot(
        history.history["val_accuracy"],
        label=name
    )

plt.title("Hyperparameter Tuning - Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid()
plt.show()


# ==========================================
# 8. PLOT LOSS COMPARISON
# ==========================================

plt.figure(figsize=(10, 5))

for name, history in histories.items():

    plt.plot(
        history.history["val_loss"],
        label=name
    )

plt.title("Hyperparameter Tuning - Loss")
plt.xlabel("Epochs")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid()
plt.show()


# ==========================================
# 9. FIND BEST CONFIGURATION
# ==========================================

best_result = max(
    results,
    key=lambda x: x["Accuracy"]
)

print("\n========== BEST CONFIGURATION ==========")

print("Best Experiment:", best_result["Experiment"])
print("Learning Rate:", best_result["Learning Rate"])
print("Batch Size:", best_result["Batch Size"])
print("Number of Layers:", best_result["Layers"])
print("Test Accuracy:", best_result["Accuracy"], "%")

print("\nHyperparameter tuning completed!")


Training data: (60000, 28, 28, 1)
Testing data: (10000, 28, 28, 1)

Running: Experiment_1
Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 56s 30ms/step - accuracy: 0.9055 - loss: 0.3239 - val_accuracy: 0.9852 - val_loss: 0.0560
Epoch 2/5
 807/1688 ━━━━━━━━━━━━━━━━━━━━ 25s 29ms/step - accuracy: 0.9843 - loss: 0.0522